In [ ]:
#An example for RNN, with Keras
from pandas import read_csv
import numpy as np
from sklearn.preprocessing import MinMaxScaler

from keras.models import Sequential
from keras.layers import Dense, SimpleRNN

from math import sqrt
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

In [ ]:
#Reading data and normalizing the data test sets
df = read_csv('MSFT.csv')
nbRows = df.shape[0]
nbCols = df.shape[1]

print(f'The data contains {nbCols} columns rows and {nbRows} rows.')

In [ ]:
data = np.array(df['Close'].values.astype('float32'))
data = data.reshape(df.shape[0],1)

scaler = MinMaxScaler()
data = scaler.fit_transform(data).flatten()

In [ ]:
#Splitting the data into training and test
split_percent = 0.8
split = int(nbRows*split_percent)
train_data = data[range(split)]
test_data = data[split:]

#Preparing the input X and target Y for the train and test sets
time_steps=10
Y_ind   = list(np.arange(time_steps-1, len(train_data), time_steps))
All_ind = list(range(time_steps*len(Y_ind)))
X_ind = list(set(All_ind)-set(Y_ind))

#On peut aussi construire X_ind de la manière suivante
X_ind2 = All_ind
for ind in Y_ind:
  X_ind2.remove(ind)  

In [ ]:
X_train = train_data[X_ind]
print(X_train.shape)
X_train = np.reshape(X_train,(len(Y_ind),(time_steps-1),1))
print(X_train.shape)
Y_train = train_data[Y_ind]

Y_ind   = list(np.arange(time_steps-1, len(test_data), time_steps))
All_ind = list(range(time_steps*len(Y_ind)))
X_ind = list(set(All_ind)-set(Y_ind))
X_test = test_data[X_ind]
X_test = np.reshape(X_test,(len(Y_ind),(time_steps-1),1))
Y_test = test_data[Y_ind]

In [ ]:
#Creating the RNN
input_shape=(time_steps-1,1)
hidden_units=3 
output_units=1 

model = Sequential()
model.add(SimpleRNN(hidden_units, input_shape=input_shape, activation='tanh'))
model.add(Dense(units=output_units, activation='linear'))

model.summary()


model.compile(loss='mean_squared_error', optimizer='adam')

model.fit(X_train, Y_train, epochs=50, batch_size=1, verbose=2)

In [ ]:

# making predictions
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

# Computing and printing errors of predictions
train_rmse = sqrt(mean_squared_error(Y_train, train_predict))
test_rmse = sqrt(mean_squared_error(Y_test, test_predict))

print('Train RMSE: %.3f RMSE' % (train_rmse))
print('Test RMSE: %.3f RMSE' % (test_rmse)) 

In [ ]:
#Plotting the predictions against the actual values

actual = np.append(Y_train, Y_test)
predictions = np.append(train_predict, test_predict)
rows = len(actual)
plt.figure(figsize=(15, 6), dpi=80)
plt.plot(range(rows), actual)
plt.plot(range(rows), predictions)
plt.axvline(x=len(Y_train), color='r')
plt.legend(['Actual', 'Predictions'])
plt.xlabel('Observation number after given time steps')
plt.ylabel('Sunspots scaled')
plt.title('Actual and Predicted Values. The Red Line Separates The Training And Test Examples')
